# Episode 33: Migration Guide

You probably didn't start here. Most teams arrive with an existing agent codebase — AG2 Classic, AutoGen 0.x, or LangChain.

**In this episode:** a side-by-side map from each of those onto the AG2 Beta API you've learned across this workshop.

## Three starting points

The migration target is always the same — `Agent`, `@tool`, `await agent.ask()`, and the Hub network for multi-agent work. What differs is where you're coming from:

| From | Key shift |
|---|---|
| AG2 Classic | `ConversableAgent` → `Agent`; `register_for_*` → `@tool`; patterns → Hub network |
| AutoGen 0.x | `AssistantAgent` + `run` → `Agent` + `await ask()` |
| LangChain | `AgentExecutor` + `Tool` → `Agent` + `@tool`; chains → tool composition |

Migrate **per agent**, not all at once — a Beta `Agent` and a classic agent can run side by side during the transition.

## From AG2 Classic

The biggest conceptual change: a tool no longer belongs to a pair of agents (one to call, one to execute). It's a plain function the agent holds.

```python
# CLASSIC — two agents, register on each side
from autogen import ConversableAgent, LLMConfig

llm_config = LLMConfig({"model": "gpt-4o-mini"})
assistant = ConversableAgent("assistant", llm_config=llm_config)
user = ConversableAgent("user", human_input_mode="NEVER")

def get_weather(city: str) -> str:
    return f"{city}: 21C, sunny."

assistant.register_for_llm(description="Weather")(get_weather)
user.register_for_execution()(get_weather)
user.run(assistant, message="Weather in Paris?", max_turns=3)
```

```python
# BETA — one agent, one tool
from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig

def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"{city}: 21C, sunny."

agent = Agent("assistant", config=OpenAIConfig(model="gpt-4.1-mini"),
              tools=[get_weather])
reply = await agent.ask("Weather in Paris?")
```

Multi-agent maps too: classic `RoundRobinPattern` → Hub **discussion** adapter; `DefaultPattern` + `OnCondition` → Hub **workflow** adapter; `AutoPattern` → the **coordinator pattern** (Episodes 10–15).

## From AutoGen 0.x

AutoGen 0.x already has an `AssistantAgent`. The migration is mostly the run API and config.

```python
# AUTOGEN 0.x
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

model = OpenAIChatCompletionClient(model="gpt-4o-mini")
agent = AssistantAgent("assistant", model_client=model, tools=[get_weather])
result = await agent.run(task="Weather in Paris?")
```

```python
# BETA
from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig

agent = Agent("assistant", config=OpenAIConfig(model="gpt-4.1-mini"),
              tools=[get_weather])
reply = await agent.ask("Weather in Paris?")
```

`run(task=...)` → `ask(...)`; the model client becomes an `OpenAIConfig`. Tools are already plain callables — they carry over unchanged.

## From LangChain

LangChain wraps tools and an executor. Beta drops the wrappers — type hints and a docstring *are* the schema.

```python
# LANGCHAIN
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import Tool

weather = Tool(name="get_weather", func=get_weather, description="Get weather")
executor = AgentExecutor(agent=create_tool_calling_agent(llm, [weather], prompt),
                         tools=[weather])
executor.invoke({"input": "Weather in Paris?"})
```

```python
# BETA — the @tool decorator reads the signature + docstring
from autogen.beta import Agent, tool
from autogen.beta.config import OpenAIConfig

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"{city}: 21C, sunny."

agent = Agent("assistant", config=OpenAIConfig(model="gpt-4.1-mini"),
              tools=[get_weather])
reply = await agent.ask("Weather in Paris?")
```

LangChain *chains* — fixed sequences of steps — become either tool composition inside one agent, or a Hub **workflow** graph when the steps are themselves agents.

## The destination, running

Every "BETA" block above lands on the same shape. Here it is for real — the end state of all three migrations, executed:

In [1]:
from dotenv import load_dotenv

load_dotenv()

from autogen.beta import Agent, tool
from autogen.beta.config import OpenAIConfig


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"{city}: 21C, sunny."


agent = Agent(
    "assistant",
    prompt="Answer weather questions using your tool.",
    config=OpenAIConfig(model="gpt-4.1-mini", temperature=0),
    tools=[get_weather],
)

reply = await agent.ask("What's the weather in Paris?")
print(reply.body)


The weather in Paris is currently 21°C and sunny.


## Migration checklist

1. **Config** — model configs become `OpenAIConfig` / `AnthropicConfig` / etc.
2. **Agents** — every agent class becomes `Agent(name, prompt=..., config=...)`.
3. **Tools** — drop the registration/wrapper boilerplate; a typed function with a docstring, in `tools=[...]`.
4. **Run** — every run/invoke/initiate-chat call becomes `await agent.ask(...)`; continue with `reply.ask(...)`.
5. **Multi-agent** — map patterns to Hub adapters (Episodes 10–15) or `agent.as_tool()` delegation (Episode 4).
6. **Test as you go** — wrap each migrated agent in `TestConfig` (Episode 27) so you can verify behavior without API calls.

## What's next

That's the whole migration. The final episode, 34, looks **forward** — where the AG2 ecosystem is heading and where to go next.